# Information extraction

Docling extracts selected structured fields from a source using an explicit target.
A target separates original JSON Schema validation from tagged model guidance and instructions.
Results contain ordered items with page or document scope; public chunk inputs are not required.
This notebook uses the existing default NuExtract2 Transformers preset; it has not been rerun as live-model evidence.


In [ ]:
%pip install -q docling[vlm]  # Install the Docling package with VLM support

In [ ]:
from IPython import display
from pydantic import BaseModel, Field
from rich import print

In this notebook, we will work with an example input image — let's quickly inspect it:

In [ ]:
file_path = (
    "https://upload.wikimedia.org/wikipedia/commons/9/9f/Swiss_QR-Bill_example.jpg"
)
display.HTML(f"<img src='{file_path}' height='1000'>")

## Defining the extractor

Let's first define our extractor:

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.extraction import ExtractionTarget, ExtractionTemplate
from docling.document_extractor import DocumentExtractor

extractor = DocumentExtractor(allowed_formats=[InputFormat.IMAGE, InputFormat.PDF])

The following targets demonstrate native caller templates, explicit JSON Schema, and Pydantic-derived schemas.


## Using a tagged native template


In [ ]:
result = extractor.extract(
    source=file_path,
    target=ExtractionTarget(
        template=ExtractionTemplate(
            format="nuextract", value={"bill_no": "string", "total": "number"}
        ),
        instructions="Copy the invoice identifier exactly",
    ),
)
for item in result.items:
    print(item.scope, item.extracted_data, item.validation_status, item.errors)

## Using an explicit output schema


In [ ]:
target = ExtractionTarget(
    output_schema={
        "type": "object",
        "properties": {"bill_no": {"type": "string"}, "total": {"type": "number"}},
        "required": ["bill_no", "total"],
    }
)
result = extractor.extract(source=file_path, target=target)
print(result.items)

## Using a Pydantic-derived output schema


First we define the Pydantic model we want to use

In [ ]:
from typing import Optional


class Invoice(BaseModel):
    bill_no: str = Field(
        examples=["A123", "5414"]
    )  # provide some examples, but no default value
    total: float = Field(
        default=10, examples=[20]
    )  # provide some examples and a default value
    tax_id: Optional[str] = Field(default=None, examples=["1234567890"])

`from_pydantic()` transfers JSON Schema only. NuExtract converts supported schema types into native guidance; Python validators are not transferred.


In [ ]:
result = extractor.extract(
    source=file_path,
    target=ExtractionTarget.from_pydantic(Invoice),
)
print(result.items)

Supply caller guidance alongside the original schema. Native NuExtract guidance uses types; generic Granite/Lift targets instead accept `example_json` values as illustrations, without schema inference.


In [ ]:
result = extractor.extract(
    source=file_path,
    target=ExtractionTarget.from_pydantic(
        Invoice,
        template=ExtractionTemplate(
            format="nuextract",
            value={"bill_no": "string", "total": "number", "tax_id": "string"},
        ),
        instructions="Read the invoice and tax identifiers exactly",
    ),
)
print(result.items)

### Advanced Pydantic model

Besides a flat template, we can in principle use any Pydantic model, thus leveraging reuse and being able to capture
hierarchies:

In [ ]:
class Contact(BaseModel):
    name: Optional[str] = Field(default=None, examples=["Smith"])
    address: str = Field(default="123 Main St", examples=["456 Elm St"])
    postal_code: str = Field(default="12345", examples=["67890"])
    city: str = Field(default="Anytown", examples=["Othertown"])
    country: Optional[str] = Field(default=None, examples=["Canada"])


class ExtendedInvoice(BaseModel):
    bill_no: str = Field(
        examples=["A123", "5414"]
    )  # provide some examples, but not the actual value of the test sample
    total: float = Field(
        default=10, examples=[20]
    )  # provide a default value and some examples
    garden_work_hours: int = Field(default=1, examples=[2])
    sender: Contact = Field(default=Contact(), examples=[Contact()])
    receiver: Contact = Field(default=Contact(), examples=[Contact()])

In [ ]:
result = extractor.extract(
    source=file_path,
    target=ExtractionTarget.from_pydantic(ExtendedInvoice),
)
print(result.items)

### Validating and loading the extracted data

Inspect `validation_status` and item errors before loading data into Python. The original JSON Schema is already validated by Docling; this additional Pydantic step creates a Python object.


In [ ]:
item = result.items[0]
assert item.validation_status == "passed", item.errors
invoice = ExtendedInvoice.model_validate(item.extracted_data)
print(invoice)

This way, we can get from completely unstructured data to a very structured and developer-friendly representation:

In [ ]:
print(
    f"Invoice #{invoice.bill_no} was sent by {invoice.sender.name} "
    f"to {invoice.receiver.name} at {invoice.sender.address}."
)